# RevAI (PS 14): Lecture2Revision — Transformer + LSTM Pipeline
This notebook implements the hybrid **Transformer + BiLSTM** architecture for combining multiple lecture transcripts on the same topic, extracting unified revision notes, and generating adaptive flashcards.

In [ ]:
import re
import numpy as np
import torch
import torch.nn as nn
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from sklearn.cluster import AgglomerativeClustering

print('PyTorch version:', torch.__version__)

## 1. Define Hybrid Transformer + BiLSTM Architecture
The Transformer (`all-MiniLM-L6-v2`) encodes semantic meaning at the sentence level, while the Bidirectional LSTM models temporal flow across sequential lecture statements.

In [ ]:
class LectureSequenceLSTM(nn.Module):
    def __init__(self, embedding_dim=384, hidden_dim=128):
        super(LectureSequenceLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.1
        )
        self.attention = nn.Linear(hidden_dim * 2, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        importance_scores = self.sigmoid(self.attention(lstm_out)).squeeze(-1)
        return lstm_out.squeeze(0), importance_scores.squeeze(0)

embedder = SentenceTransformer('all-MiniLM-L6-v2')
summarizer = pipeline('summarization', model='sshleifer/distilbart-cnn-12-6')
lstm_model = LectureSequenceLSTM()
lstm_model.eval()

## 2. Sample Multi-Video Lecture Inputs on the Same Topic
Below we simulate transcripts from two separate lectures covering **Neural Networks & Backpropagation**.

In [ ]:
topic = 'Neural Networks and Backpropagation'

lecture_video_1 = (
    'Artificial neural networks are computing systems inspired by biological neural networks. '
    'Each neuron receives inputs, multiplies them by weights, adds a bias, and passes the result through an activation function. '
    'ReLU is the most widely used activation function in hidden layers because it mitigates the vanishing gradient problem. '
    'Forward propagation is the process of passing input data through the network to generate a prediction.'
)

lecture_video_2 = (
    'During training, a loss function measures the discrepancy between predicted outputs and actual target labels. '
    'Backpropagation is an algorithm that calculates the gradient of the loss function with respect to each weight using the chain rule. '
    'Gradient descent updates the weights in the opposite direction of the gradient to minimize error. '
    'If gradients become extremely small in deep networks, earlier layers learn very slowly, which is known as the vanishing gradient problem.'
)

lectures = [lecture_video_1, lecture_video_2]

## 3. Multi-Video Alignment, Note Synthesis & Flashcard Generation

In [ ]:
sentences = []
sources = []
for idx, text in enumerate(lectures):
    sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    sentences.extend(sents)
    sources.extend([f'Video {idx+1}'] * len(sents))

# 1. Transformer Encoding
emb_np = embedder.encode(sentences, convert_to_numpy=True)
emb_tensor = torch.tensor(emb_np, dtype=torch.float32).unsqueeze(0)

# 2. BiLSTM Sequence Scoring
with torch.no_grad():
    _, lstm_scores = lstm_model(emb_tensor)
    scores_np = lstm_scores.numpy()

# 3. Semantic Clustering across Videos
clustering = AgglomerativeClustering(n_clusters=2)
labels = clustering.fit_predict(emb_np)

print('=== UNIFIED REVISION NOTES ===')
for c_id in range(2):
    c_indices = np.where(labels == c_id)[0]
    combined_text = ' '.join([sentences[i] for i in c_indices])
    v_sources = sorted(list(set([sources[i] for i in c_indices])))
    summary = summarizer(combined_text, max_length=60, min_length=20, do_sample=False)[0]['summary_text']
    print(f'\nModule {c_id+1} (Sources: {", ".join(v_sources)}):')
    print('Summary:', summary)

print('\n=== GENERATED FLASHCARDS ===')
for i, sent in enumerate(sentences):
    print(f'Q{i+1} [{sources[i]}]: Explain the concept described by: "{sent[:35]}..."')
    print(f'A{i+1}: {sent}\n')